# 1. CAPA DO PROJETO

## Otimização do Tráfego Urbano: Uma Abordagem de IA para Cidades Inteligentes

### Desvendando os Padrões de Mobilidade para um Futuro Mais Fluido

**Autores:** [Nomes dos Integrantes]

**Grupo:** GRUPO 6

**Disciplina:** [Nome da Disciplina]

**Título do Desafio:** Como fazer para melhorar o trânsito das cidades

**Objetivo do Projeto:** Desenvolver um projeto de IA aplicada capaz de analisar padrões de trânsito, identificar gargalos urbanos, detectar horários críticos, prever comportamento do tráfego, gerar insights estratégicos para mobilidade urbana e propor soluções inteligentes para redução de congestionamentos.

# 2. CONTEXTUALIZAÇÃO DO PROBLEMA

A mobilidade urbana é um desafio crescente nas grandes cidades, impactando diretamente a qualidade de vida dos cidadãos, a economia e o meio ambiente. Congestionamentos frequentes resultam em perda de tempo, aumento do consumo de combustível, poluição e estresse. A aplicação de Inteligência Artificial (IA) surge como uma ferramenta poderosa para analisar a complexidade do tráfego, identificar padrões e propor soluções inovadoras para otimizar o fluxo e tornar as cidades mais inteligentes e sustentáveis.

# 3. OBJETIVO GERAL

Desenvolver um sistema de IA para análise e previsão de tráfego urbano, visando a melhoria da mobilidade nas cidades através da identificação de padrões, gargalos e horários críticos, e a proposição de soluções inteligentes para a redução de congestionamentos.

# 4. OBJETIVOS ESPECÍFICOS

- Analisar padrões de trânsito utilizando dados históricos.
- Identificar gargalos urbanos e pontos de congestionamento.
- Detectar horários de pico e eventos que impactam o fluxo de tráfego.
- Prever o comportamento futuro do tráfego.
- Gerar insights estratégicos para planejamento urbano e políticas públicas.
- Propor soluções baseadas em IA para otimização da mobilidade.

# 5. IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
# Instalação automática de dependências
!pip -q install xgboost missingno plotly folium gdown

# Bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from IPython.display import display
import xgboost as xgb
import missingno as msno
import folium
import warnings
import os

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Bibliotecas carregadas com sucesso.")


# 6. CARREGAMENTO DO DATASET

In [ ]:
# ==========================================================
# CARREGAMENTO AUTOMÁTICO DO DATASET
# ==========================================================

import gdown

FILE_ID = "1aCQg7WYuiUEwvDOtsuuyyU_CB2SRxuVw"
OUTPUT = "dataset_trafego.csv"

if not os.path.exists(OUTPUT):
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, OUTPUT, quiet=False)

file_path = OUTPUT

try:
    # Tentativa com separador ';'
    df = pd.read_csv(file_path, sep=';', encoding='latin1', nrows=50000)

    # Caso carregue apenas uma coluna, tenta outro separador
    if len(df.columns) == 1:
        df = pd.read_csv(file_path, sep=',', encoding='latin1', nrows=50000)

    print('Dataset carregado com sucesso.')
    print(f'Formato do dataset: {df.shape}')

    print("\nColunas encontradas:")
    print(df.columns.tolist())

    display(df.head())

except Exception as e:
    print(f"Erro ao carregar dataset: {e}")


# 7. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)

In [ ]:
if 'df' in locals():
    print('\nValores ausentes por coluna:')
    print(df.isnull().sum())

    # Visualização de valores ausentes
    msno.bar(df, figsize=(10, 5), color='steelblue')
    plt.title('Gráfico de Barras de Valores Ausentes')
    plt.show()
else:
    print('DataFrame não carregado.')

In [ ]:
if 'df' in locals():

    # Identificação automática da coluna de data
    possible_date_cols = [c for c in df.columns if 'data' in c.lower()]

    if len(possible_date_cols) > 0:
        date_col = possible_date_cols[0]

        df[date_col] = pd.to_datetime(
            df[date_col],
            errors='coerce',
            dayfirst=True
        )

        df['hora'] = df[date_col].dt.hour
        df['dia_da_semana'] = df[date_col].dt.day_name()

        plt.figure(figsize=(12,6))
        sns.countplot(x='hora', data=df, palette='viridis')
        plt.title('Volume de Tráfego por Hora do Dia')
        plt.xlabel('Hora')
        plt.ylabel('Quantidade')
        plt.show()

    else:
        print("Nenhuma coluna de data encontrada.")

else:
    print('DataFrame não carregado.')


# 8. PRÉ-PROCESSAMENTO

In [ ]:
if 'df' in locals():

    # Tentativa de localizar automaticamente colunas relevantes
    velocidade_col = None
    lat_col = None
    lon_col = None

    for c in df.columns:
        if 'veloc' in c.lower():
            velocidade_col = c
        if 'lat' in c.lower():
            lat_col = c
        if 'lon' in c.lower():
            lon_col = c

    if velocidade_col and lat_col and lon_col:

        df_ml = df.dropna(subset=[velocidade_col, lat_col, lon_col]).copy()

        # Conversões numéricas
        df_ml[velocidade_col] = pd.to_numeric(df_ml[velocidade_col], errors='coerce')
        df_ml[lat_col] = pd.to_numeric(df_ml[lat_col], errors='coerce')
        df_ml[lon_col] = pd.to_numeric(df_ml[lon_col], errors='coerce')

        df_ml = df_ml.dropna()

        if 'Sentido' in df_ml.columns:
            df_ml = pd.get_dummies(df_ml, columns=['Sentido'], drop_first=True)

        features = ['hora', lat_col, lon_col]

        for c in df_ml.columns:
            if 'Sentido_' in c:
                features.append(c)

        X = df_ml[features]
        y = df_ml[velocidade_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42
        )

        print(f'Dados preparados para ML.')
        print(f'Treino: {X_train.shape}')
        print(f'Teste: {X_test.shape}')

    else:
        print("Colunas necessárias não encontradas.")

else:
    print('DataFrame não carregado.')


# 9. MODELAGEM E AVALIAÇÃO

In [ ]:
if 'X_train' in locals():

    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f'R2 Score: {r2:.4f}')
    print(f'RMSE: {rmse:.4f}')

    # Visualização Real vs Previsto
    plt.figure(figsize=(10,6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.xlabel('Valores Reais')
    plt.ylabel('Valores Preditos')
    plt.title('Real vs Predito')
    plt.show()

else:
    print('Dados de treino não disponíveis.')


# 10. DASHBOARD EXECUTIVO (Plotly)

In [ ]:
if 'df' in locals():

    if 'hora' in df.columns and 'dia_da_semana' in df.columns:

        fig = px.histogram(
            df,
            x='hora',
            color='dia_da_semana',
            barmode='group',
            title='Distribuição de Tráfego por Hora e Dia da Semana'
        )

        fig.update_layout(template='plotly_dark')
        fig.show()

    # Identificação automática de coluna de endereço/região
    endereco_col = None

    for c in df.columns:
        if 'endere' in c.lower() or 'regiao' in c.lower():
            endereco_col = c
            break

    if endereco_col:

        top_enderecos = (
            df[endereco_col]
            .value_counts()
            .head(10)
            .reset_index()
        )

        top_enderecos.columns = ['Endereco', 'count']

        fig2 = px.bar(
            top_enderecos,
            x='Endereco',
            y='count',
            title='Top 10 Regiões com Maior Volume'
        )

        fig2.update_layout(template='plotly_dark')
        fig2.show()

else:
    print('DataFrame não carregado.')


# CONCLUSÕES E RECOMENDAÇÕES

1. **Otimização de Semáforos:** Ajustar tempos com base nos picos identificados.
2. **Rotas Alternativas:** Implementar sinalização dinâmica para desviar tráfego em horários críticos.
3. **Cidades Inteligentes:** Utilizar este modelo preditivo para planejar expansões urbanas.


# 📊 Visualizações Analíticas e Interpretação dos Resultados

Nesta seção são apresentados gráficos analíticos e dashboards interativos desenvolvidos para identificar padrões de mobilidade urbana, gargalos de tráfego e tendências temporais.

Cada gráfico possui interpretação executiva e técnica para apoiar tomadas de decisão em cidades inteligentes.



# 🔥 Heatmap de Correlação

O heatmap mostra o grau de correlação entre variáveis numéricas do dataset.

## O que este gráfico significa?
- Correlações positivas indicam que duas variáveis aumentam juntas.
- Correlações negativas indicam comportamento inverso.
- Valores próximos de 1 ou -1 representam relações fortes.
- Valores próximos de 0 indicam pouca relação linear.

## Objetivo no contexto do trânsito
Identificar fatores que influenciam diretamente:
- velocidade do tráfego;
- intensidade de congestionamentos;
- padrões espaciais e temporais.


In [ ]:

# ==========================================================
# HEATMAP DE CORRELAÇÃO
# ==========================================================

numeric_df = df.select_dtypes(include=['int64', 'float64']).copy()

if numeric_df.shape[1] > 1:

    corr = numeric_df.corr()

    plt.figure(figsize=(14,10))
    sns.heatmap(
        corr,
        annot=True,
        cmap='coolwarm',
        fmt='.2f'
    )

    plt.title('Heatmap de Correlação das Variáveis Numéricas')
    plt.show()

else:
    print("Poucas colunas numéricas disponíveis para correlação.")



# ⏰ Distribuição do Tráfego por Hora

Este gráfico mostra os horários de maior fluxo de veículos.

## Interpretação
- Picos representam horários críticos.
- Permite identificar horários de congestionamento.
- Auxilia no planejamento urbano e semafórico.

## Aplicação prática
Os horários críticos podem orientar:
- sincronização inteligente de semáforos;
- definição de rotas alternativas;
- estratégias de mobilidade urbana.


In [ ]:

# ==========================================================
# HISTOGRAMA DE TRÁFEGO POR HORA
# ==========================================================

if 'hora' in df.columns:

    plt.figure(figsize=(14,6))

    sns.histplot(
        df['hora'].dropna(),
        bins=24,
        kde=True
    )

    plt.title('Distribuição do Fluxo de Tráfego por Hora')
    plt.xlabel('Hora do Dia')
    plt.ylabel('Frequência')
    plt.show()

else:
    print("Coluna 'hora' não encontrada.")



# 🚗 Boxplot da Velocidade do Tráfego

O boxplot apresenta a distribuição estatística da velocidade dos veículos.

## O que analisar?
- Mediana;
- dispersão;
- valores extremos;
- possíveis anomalias.

## Importância
Ajuda a detectar:
- congestionamentos severos;
- comportamento anormal do trânsito;
- regiões críticas.


In [ ]:

# ==========================================================
# BOXPLOT DE VELOCIDADE
# ==========================================================

velocidade_col = None

for c in df.columns:
    if 'veloc' in c.lower():
        velocidade_col = c
        break

if velocidade_col:

    plt.figure(figsize=(12,6))

    sns.boxplot(
        x=df[velocidade_col]
    )

    plt.title('Distribuição Estatística da Velocidade')
    plt.xlabel('Velocidade')

    plt.show()

else:
    print("Coluna de velocidade não encontrada.")



# 📍 Relação Geográfica do Trânsito

Este gráfico de dispersão representa os pontos geográficos do tráfego urbano.

## Interpretação
- Regiões mais densas representam maior concentração de fluxo.
- Permite identificar corredores urbanos críticos.
- Facilita análises espaciais.

## Aplicação
Pode ser usado em:
- planejamento viário;
- expansão urbana;
- otimização de rotas.


In [ ]:

# ==========================================================
# SCATTER GEOGRÁFICO
# ==========================================================

lat_col = None
lon_col = None

for c in df.columns:
    if 'lat' in c.lower():
        lat_col = c
    if 'lon' in c.lower():
        lon_col = c

if lat_col and lon_col:

    fig = px.scatter_mapbox(
        df.sample(min(3000, len(df))),
        lat=lat_col,
        lon=lon_col,
        zoom=10,
        title='Mapa Geográfico do Fluxo de Tráfego'
    )

    fig.update_layout(mapbox_style='open-street-map')
    fig.show()

else:
    print("Colunas geográficas não encontradas.")



# 🤖 Importância das Variáveis no Modelo de IA

Este gráfico mostra quais variáveis tiveram maior impacto nas previsões do modelo.

## Interpretação
- Quanto maior a barra, maior a influência da variável.
- Ajuda na explicabilidade da IA.
- Permite compreender os fatores mais relevantes do trânsito.

## Benefícios
- transparência do modelo;
- apoio à tomada de decisão;
- inteligência analítica.


In [ ]:

# ==========================================================
# IMPORTÂNCIA DAS FEATURES
# ==========================================================

if 'model' in locals() and 'X_train' in locals():

    importance = pd.DataFrame({
        'Variavel': X_train.columns,
        'Importancia': model.feature_importances_
    })

    importance = importance.sort_values(
        by='Importancia',
        ascending=False
    )

    plt.figure(figsize=(12,6))

    sns.barplot(
        data=importance,
        x='Importancia',
        y='Variavel'
    )

    plt.title('Importância das Variáveis no Modelo')
    plt.show()

else:
    print("Modelo não treinado.")



# ✅ Conclusões Analíticas

Com base nas análises realizadas, foi possível:

- identificar horários críticos;
- detectar padrões de congestionamento;
- compreender variáveis relevantes;
- construir modelos preditivos;
- gerar insights para cidades inteligentes.

## Potenciais aplicações
- semáforos inteligentes;
- roteamento dinâmico;
- previsão de congestionamentos;
- gestão inteligente da mobilidade urbana;
- apoio à administração pública.
